# Lab 01: Sampling and Probability
This lab revisits the lecture *Data Sampling and Probability*.

| Task | Topic |
|---|---|
| 0 | numpy warm-up (just run and read) |
| 1 | Binomial probability |
| 2 | Multinomial probability |
| 3 | Simulation and expected value |
| 4 | Linearity of expectation |
| 5 | Sampling bias: a mini *Literary Digest* |

Fill in the cells marked `# YOUR CODE HERE` and run the test cell below each one.

In [1]:
import numpy as np
from math import comb, factorial

## Setting
We draw $n = 80$ times **with replacement** from a population with 3 categories.
The proportions are $p_1 = 0.15$, $p_2 = 0.3$, $p_3 = 0.55$.
We count how often each category occurs, which gives the random vector $X = (X_1, X_2, X_3)$.
So $X$ follows a **multinomial** distribution.

In [2]:
n = 80
p = np.array([0.15, 0.3, 0.55])

## Task 0: numpy warm-up (nothing to solve)
Run the cell below and read the output carefully. You will need these four tricks in the next tasks.

In [3]:
rng = np.random.default_rng(seed=0)

# 5 repetitions of "draw 80 times and count the categories"
demo = rng.multinomial(n, p, size=5)
print("The result is a table with 5 rows (repetitions) and 3 columns (categories):")
print(demo)
print("shape:", demo.shape)

print("Each row sums to n:", demo.sum(axis=1))
print("Mean of each column (axis=0 means 'go down the rows'):", demo.mean(axis=0))
print("Only column 0, i.e. the counts of category 1:", demo[:, 0])

The result is a table with 5 rows (repetitions) and 3 columns (categories):
[[13 21 46]
 [ 7 17 56]
 [15 28 37]
 [13 26 41]
 [12 30 38]]
shape: (5, 3)
Each row sums to n: [80 80 80 80 80]
Mean of each column (axis=0 means 'go down the rows'): [12.  24.4 43.6]
Only column 0, i.e. the counts of category 1: [13  7 15 13 12]


## Task 1: Binomial probability
Recall the marble example from the lecture: 60% of the marbles are blue. We draw 7 times with replacement.

Compute $P(\text{4 blue, 3 not blue}) = \binom{7}{4}\, 0.6^4 \, 0.4^3$.

Use `comb(n, k)` to compute $\binom{n}{k}$ and store the result in `P_binom`.

In [4]:
# YOUR CODE HERE

P_binom = comb(7, 4) * pow(0.6, 4) * pow(0.4, 3) 
print(P_binom)

0.29030400000000006


In [5]:
assert 0 < P_binom < 1, "A probability must be between 0 and 1"
assert np.isclose(P_binom, 0.290304)


## Task 2: Multinomial probability
Using `n` and `p` from the setting, compute the probability that category 1 occurs exactly 7 times,
category 2 occurs 22 times, and category 3 occurs 51 times:

$$P = \frac{n!}{k_1!\,k_2!\,k_3!}\; p_1^{k_1}\, p_2^{k_2}\, p_3^{k_3}$$

Store the result in `P`.

*Hint:* `p ** k` with `k = np.array([7, 22, 51])` computes all three powers at once, and `np.prod` multiplies them together.

In [6]:
k = np.array([7, 22, 51])


# YOUR CODE HERE

divider = 1

for kx in k:
    divider *= factorial(kx)

amount = (factorial(n) / divider)

prop = np.prod(p ** k)

P = amount * prop
print(P)

0.0025043159900098666


In [7]:
assert 0 < P < 1, "A probability must be between 0 and 1"
assert np.isclose(P, 0.0025043159900098666)


## Task 3: Simulation and expected value
For a multinomial, $\mathbb{E}[X_i] = n \cdot p_i$, just like for the binomial.

1. Compute the analytical means $n \cdot p$ for all three categories at once and store them in `analytical_mean`.
2. Use `rng.multinomial` to simulate `100` repetitions (`sim_100`) and `10000` repetitions (`sim_10000`).
3. Compute the column means of both simulations: `mean_100` and `mean_10000`.
4. Compute the absolute errors `error_100 = |mean_100 - analytical_mean|`, and the same for `error_10000`. Use `np.abs`.

Which simulation is closer to the analytical mean? Why?

In [16]:
rng = np.random.default_rng(seed=57238464)

# YOUR CODE HERE

analytical_mean = n * p

sim_100 = rng.multinomial(n, p, size=100)
mean_100 = sim_100.mean(axis=0)
error_100 = np.abs(mean_100 - analytical_mean)

sim_10000 = rng.multinomial(n, p, size=10000)
mean_10000 = sim_10000.mean(axis=0)
error_10000 = np.abs(mean_10000 - analytical_mean)

print("analytical:", analytical_mean)
print("mean_100:  ", mean_100, " error:", error_100)
print("mean_10000:", mean_10000, " error:", error_10000)

analytical: [12. 24. 44.]
mean_100:   [11.93 24.15 43.92]  error: [0.07 0.15 0.08]
mean_10000: [11.9992 23.9722 44.0286]  error: [0.0008 0.0278 0.0286]


In [17]:
assert sim_100.shape == (100, 3), "sim_100 should have 100 rows and 3 columns"
assert sim_10000.shape == (10000, 3), "sim_10000 should have 10000 rows and 3 columns"
assert mean_100.shape == (3,), "Did you use axis=0?"
assert np.allclose(analytical_mean, [12, 24, 44])
# tolerance-based, so any seed or draw order passes (about 4.5 standard errors)
assert np.allclose(mean_100, analytical_mean, atol=2.0)
assert np.allclose(mean_10000, analytical_mean, atol=0.2)
assert np.allclose(error_100, np.abs(mean_100 - analytical_mean))
assert np.allclose(error_10000, np.abs(mean_10000 - analytical_mean))


## Task 4: Linearity of expectation
We want $\mathbb{E}[2X_1 - X_2 + 5X_3]$.

1. **Analytically:** use linearity, $\mathbb{E}[\sum_i c_i X_i] = \sum_i c_i\, \mathbb{E}[X_i]$, together with `analytical_mean` from Task 3. Store the result in `exp_value_analytical`.
2. **Empirically:** for every row of `sim_10000`, compute $2X_1 - X_2 + 5X_3$ and take the mean. Store it in `exp_value_sim`.
   *Hint:* `sim_10000[:, 0]` is the column $X_1$.

Why are the two values close but not exactly equal?

In [21]:
# YOUR CODE HERE

exp_value_analytical = np.sum(np.array([2, -1, 5]) * analytical_mean)

exp_value_rows = 2 * sim_10000[:, 0] - sim_10000[:, 1] + 5 * sim_10000[:, 2]

exp_value_sim = exp_value_rows.mean(axis=0)



In [22]:
exp_value_analytical, exp_value_sim

(np.float64(220.0), np.float64(220.1692))

In [23]:
assert np.ndim(exp_value_sim) == 0, "exp_value_sim should be a single number. Did you take the mean?"
assert np.isclose(exp_value_analytical, 220)
assert abs(exp_value_sim - 220) < 1.0


## Task 5: Sampling bias, a mini *Literary Digest*
In 1936, the *Literary Digest* surveyed 10 million people from phone books and country club lists and predicted that Roosevelt would lose.
We recreate that situation with a made-up population of 100,000 voters.

- `income`: the income of each voter
- `votes_roosevelt`: 1 if the voter votes for Roosevelt, 0 otherwise

Wealthier voters are less likely to vote for Roosevelt. Run the next cell (it builds the population) and don't change it.

In [24]:
pop_rng = np.random.default_rng(seed=1936)
income = pop_rng.lognormal(mean=10, sigma=0.8, size=100_000)
rich = income > np.quantile(income, 0.7)          # the richest 30%
votes_roosevelt = np.where(rich, pop_rng.random(100_000) < 0.43,
                                 pop_rng.random(100_000) < 0.69).astype(int)
print("population size:", len(votes_roosevelt))

population size: 100000


1. Compute the true share of Roosevelt voters in the whole population: `true_share`. *Hint:* the mean of a 0/1 array is a share.
2. Draw a **simple random sample** of 1000 voters, i.e. uniformly at random **without replacement**. Use `rng.choice(votes_roosevelt, size=1000, replace=False)` and store the share in `random_share`.
3. The Digest's **sampling frame** contained only wealthy people. Build `frame = votes_roosevelt[income > threshold]` with the given `threshold`.
   Then draw random samples of size 1000 and 10000 **from the frame** and store their shares in `biased_share_1000` and `biased_share_10000`.

In [37]:
rng = np.random.default_rng(seed=42)
threshold = np.quantile(income, 0.7)

# YOUR CODE HERE

true_share = votes_roosevelt.mean()

random_share = rng.choice(votes_roosevelt, size=1000, replace=False).mean()

frame = votes_roosevelt[rich]

biased_share_1000 = rng.choice(frame, size=1000, replace=False).mean()

biased_share_10000 = rng.choice(frame, size=10000, replace=False).mean()

print("true share:         ", true_share)
print("random sample (1000):", random_share)
print("biased sample (1000):", biased_share_1000)
print("biased sample (10000):", biased_share_10000)

true share:          0.61173
random sample (1000): 0.639
biased sample (1000): 0.449
biased sample (10000): 0.4302


In [38]:
assert 0 <= true_share <= 1 and 0 <= random_share <= 1, "Shares must be between 0 and 1"
assert np.isclose(true_share, votes_roosevelt.mean())
assert abs(random_share - true_share) < 0.06
assert len(frame) == 30_000
assert abs(biased_share_1000 - 0.43) < 0.06
assert abs(biased_share_10000 - 0.43) < 0.03


**Question:** The biased sample of 10000 is ten times larger than the random sample of 1000. Which one gives the better estimate, and why?
Which kind of bias from the lecture is this?

YOUR ANSWER HERE